# M26 lab solution: Coding agent

**Brief.** Give a deep agent a developer sub-agent and a tester sub-agent, ask it for a small Python function with tests, and log each iteration until the tests pass.

The function is the refund-amount calculator in `../data/spec.md`, whose rules come from the Larkspur handbook. The agent is built as `03a_simple_coding_agent.py` is: `create_deep_agent` with sub-agent dicts and a `FilesystemBackend`. The backend has no shell, so the agent can write files but cannot run them; this notebook runs the tests after each iteration and sends the report back, which grades the work on the environment instead of on the agent's words.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## 1. A temporary workspace

The agents see only this folder: the spec and the handbook's refunds page.

In [ ]:
import os
import shutil
import subprocess
import tempfile

workdir = Path(tempfile.mkdtemp(prefix="m26_refund_"))
shutil.copy(DATA / "spec.md", workdir / "spec.md")
shutil.copy(DATA / "larkspur" / "handbook" / "02_refunds.md", workdir / "02_refunds.md")
print("Workspace:", workdir)
print(sorted(p.name for p in workdir.iterdir()))

## 2. The deep agent: an orchestrator, a developer and a tester

Each sub-agent is a dict of `name`, `description` and `system_prompt`, as in `03a_simple_coding_agent.py`. The orchestrator reaches them through the `task` tool, so each one works in its own context and only its final message comes back. `FilesystemBackend(root_dir=workdir, virtual_mode=True)` anchors `/` at the workspace and blocks `..`; it provides no process isolation, and it cannot run commands, which is why the tests run in step 3 instead.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import MemorySaver

developer = {
    "name": "developer",
    "description": "Writes and fixes /refund.py so that it meets /spec.md. Use it for every change to the implementation.",
    "system_prompt": ("You are a careful Python developer. Read /spec.md, then write or fix /refund.py with write_file or edit_file. "
                      "Write only refund.py and never edit the tests. Given a failing test report, fix the code, not the test. "
                      "Reply with one short paragraph on what you changed."),
}
tester = {
    "name": "tester",
    "description": "Writes /test_refund.py, the pytest tests for /spec.md. Use it before the first implementation, or to fix a test that contradicts the spec.",
    "system_prompt": ("You are a test engineer. Read /spec.md and write /test_refund.py for pytest with write_file. "
                      "Import the function with `from refund import refund_amount`. Cover every worked example, every ValueError case "
                      "and every boundary day the spec lists, and compare money with pytest.approx. Never write refund.py. "
                      "Reply with the number of tests you wrote."),
}
ORCHESTRATOR_PROMPT = ("You lead a two-person team building /refund.py from /spec.md. Delegate with the task tool: "
                       "the tester writes /test_refund.py and the developer writes /refund.py. Do not write either file yourself. "
                       "You cannot run the tests; after each round the user sends you the pytest report. "
                       "When tests fail because the code is wrong, send the report to the developer. "
                       "Only when a test contradicts /spec.md, ask the tester to fix that test, and say so in your reply.")

agent = create_deep_agent(
    model=llm,
    tools=[],
    system_prompt=ORCHESTRATOR_PROMPT,
    subagents=[developer, tester],
    backend=FilesystemBackend(root_dir=workdir, virtual_mode=True),
    checkpointer=MemorySaver(),
)
print("Deep agent ready. Sub-agents:", [developer["name"], tester["name"]])

## 3. The test runner

The tests run in a fresh Python process in the workspace, with a timeout, no keyboard and no API keys in its environment. As in `python_runner.py`, that is a guard, not a sandbox: the code still runs with your user's permissions, so run this lab in a throwaway environment.

In [ ]:
def run_in_workspace(args, timeout=60):
    env = {"PATH": os.environ.get("PATH", ""), "PYTHONDONTWRITEBYTECODE": "1"}
    try:
        proc = subprocess.run([sys.executable, *args], cwd=workdir, env=env, stdin=subprocess.DEVNULL,
                              capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return None, f"timed out after {timeout} s"
    return proc.returncode, proc.stdout + proc.stderr


def run_tests():
    code_, out = run_in_workspace(["-m", "pytest", "-q", "-p", "no:cacheprovider", "test_refund.py"])
    lines = [line for line in out.splitlines() if line.strip()]
    return {"returncode": code_, "summary": lines[-1] if lines else "no output", "report": out[-4000:]}

## 4. Iterate until the tests pass, with a cap in code

The cap is a loop bound in this cell, not a request in a prompt. Each iteration logs which sub-agents the orchestrator called and the pytest summary line.

In [ ]:
MAX_ITERATIONS = 5
config = {"configurable": {"thread_id": "refund-1"}, "recursion_limit": 200}
log, seen = [], 0
message = "Build /refund.py and /test_refund.py from /spec.md."

for iteration in range(1, MAX_ITERATIONS + 1):
    result = agent.invoke({"messages": [{"role": "user", "content": message}]}, config)
    new, seen = result["messages"][seen:], len(result["messages"])
    delegated = [tc["args"].get("subagent_type") for m in new for tc in (getattr(m, "tool_calls", None) or [])
                 if tc["name"] == "task"]
    run = run_tests()
    log.append({"iteration": iteration, "sub-agents called": ", ".join(delegated) or "none",
                "files": ", ".join(sorted(p.name for p in workdir.glob("*.py"))),
                "pytest exit code": run["returncode"], "pytest summary": run["summary"]})
    print(f"Iteration {iteration}: {run['summary']}")
    if run["returncode"] == 0:
        break
    message = "The tests failed. Here is the pytest report:\n\n" + run["report"]
else:
    print(f"Stopped by the cap after {MAX_ITERATIONS} iterations.")

## 5. Results

In [ ]:
iterations = pd.DataFrame(log)
print(iterations.to_string(index=False))
for name in ("refund.py", "test_refund.py"):
    path = workdir / name
    print(f"\n===== {name} =====\n" + (path.read_text() if path.exists() else "(not written)"))

## 6. An independent check against the spec

The tester wrote the tests, so a passing run proves only that the code agrees with the tester. This cell runs the spec's seven worked examples, typed in by hand here, against the agent's function in a separate process.

In [ ]:
EXAMPLES = [("monthly", 40.0, 3, 5, 0.0), ("annual", 1200.0, 10, 20, 1200.0), ("annual", 1200.0, 30, 99.9, 1200.0),
            ("annual", 1200.0, 10, 100, 935.0), ("annual", 1200.0, 31, 0, 850.0), ("annual", 1200.0, 200, 0, 425.0),
            ("annual", 1200.0, 360, 0, 0.0)]
check = ("from refund import refund_amount\n"
         f"for args in {EXAMPLES!r}:\n"
         "    got = refund_amount(*args[:4])\n"
         "    print(args[:4], 'expected', args[4], 'got', got, 'ok' if abs(got - args[4]) < 0.005 else 'WRONG')\n")
code_, out = run_in_workspace(["-c", check], timeout=30)
print(out)

## What this shows

- The log is the brief's deliverable: one row per iteration, with the sub-agents the orchestrator called and the test result the environment returned. The loop ended either on a zero pytest exit code or on the cap.
- The deep agent never ran code. Its backend had file tools only, and the tests ran here, outside the agent, so the pass or fail it saw each round was measured, not estimated.
- A green test run and a correct function are different claims. Step 6 checks the second with examples taken from the spec rather than from the tester.
- Delete the workspace when you are done: `shutil.rmtree(workdir)`.